# 🚗 KUSRC Traffic — YOLO Fine-tuning บน Kaggle
### คู่มือฉบับมือใหม่

**ก่อนรัน Notebook นี้ ตรวจสอบให้ครบ:**

1. ✅ เพิ่ม Dataset **traffic-vehicles-yolo** เข้า Notebook แล้ว (+ Add Input)
2. ✅ เปิด **GPU** ใน Session options (Accelerator → T4 x2 หรือ P100)
3. ✅ เปิด **Internet** ใน Session options

จากนั้นกด **Run All** ได้เลย — รอประมาณ 20-40 นาที ☕

In [ ]:
# ============================================================
# Step 1: ติดตั้ง ultralytics
# ============================================================
import subprocess, sys

print('กำลังติดตั้ง ultralytics...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'ultralytics', '--quiet'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ ติดตั้ง ultralytics สำเร็จ!')
else:
    print('❌ ติดตั้งไม่สำเร็จ — ตรวจสอบว่าเปิด Internet ใน Session options แล้วหรือยัง')
    print(result.stderr)

import ultralytics
ultralytics.checks()

In [ ]:
# ============================================================
# Step 2: หา Dataset path อัตโนมัติ + สร้าง data.yaml ที่ถูกต้อง
# ============================================================
import os, yaml

# หา dataset root อัตโนมัติ
def find_dataset_root(base='/kaggle/input'):
    """ค้นหาโฟลเดอร์ที่มี data.yaml อยู่"""
    for root, dirs, files in os.walk(base):
        if 'data.yaml' in files:
            return root
    return None

dataset_root = find_dataset_root()

if dataset_root is None:
    print('❌ ไม่พบ data.yaml ใน /kaggle/input/')
    print('   → ตรวจสอบว่าเพิ่ม Dataset เข้า Notebook แล้ว (+ Add Input)')
    print()
    print('📁 โครงสร้างที่มีอยู่:')
    for root, dirs, files in os.walk('/kaggle/input/'):
        level = root.replace('/kaggle/input/', '').count(os.sep)
        if level > 3:
            continue
        print('  ' * level + '📂 ' + os.path.basename(root) + '/')
        for f in files:
            print('  ' * (level + 1) + '📄 ' + f)
    raise FileNotFoundError('ไม่พบ data.yaml')

print(f'✅ พบ Dataset ที่: {dataset_root}')

# อ่าน data.yaml เดิม
original_yaml = os.path.join(dataset_root, 'data.yaml')
with open(original_yaml) as f:
    orig_data = yaml.safe_load(f)

print(f'\n📄 data.yaml เดิม:')
for k, v in orig_data.items():
    print(f'  {k}: {v}')

# ตรวจสอบโฟลเดอร์ภาพ
print()
train_img = os.path.join(dataset_root, 'train', 'images')
val_img   = os.path.join(dataset_root, 'valid', 'images')
test_img  = os.path.join(dataset_root, 'test',  'images')

for label, path in [('train', train_img), ('valid', val_img), ('test', test_img)]:
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f'  ✅ {label}: {path} ({n} ภาพ)')
    else:
        print(f'  ❌ {label}: ไม่พบ {path}')

# สร้าง data.yaml ใหม่ที่ใช้ absolute paths
nc    = orig_data.get('nc', 4)
names = orig_data.get('names', ['bus', 'car', 'motorcycle', 'truck'])

fixed = {
    'train': train_img,
    'val':   val_img,
    'test':  test_img,
    'nc':    nc,
    'names': names,
}

yaml_path = '/kaggle/working/data_fixed.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(fixed, f, allow_unicode=True, default_flow_style=False)

print(f'\n✅ สร้าง data_fixed.yaml แล้ว:')
for k, v in fixed.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Step 3: ตรวจสอบ GPU
# ============================================================
import torch

if torch.cuda.is_available():
    gpu_name  = torch.cuda.get_device_name(0)
    gpu_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_count = torch.cuda.device_count()
    print(f'✅ GPU พร้อมใช้งาน!')
    print(f'   GPU   : {gpu_name}')
    print(f'   VRAM  : {gpu_mem:.1f} GB')
    print(f'   Count : {gpu_count}')

    # แนะนำ batch size
    if gpu_mem >= 15:
        print(f'\n💡 แนะนำ BATCH = 16 (VRAM เยอะ)')
    elif gpu_mem >= 8:
        print(f'\n💡 แนะนำ BATCH = 8')
    else:
        print(f'\n💡 แนะนำ BATCH = 4 (VRAM น้อย)')
else:
    print('⚠️  ไม่พบ GPU!')
    print('   → Session options → Accelerator → GPU T4 x2 หรือ P100')

In [ ]:
# ============================================================
# Step 4: เทรนโมเดล  (~20-40 นาที)
# ============================================================
from ultralytics import YOLO

BASE_MODEL = 'yolo11s.pt'   # n=เร็ว | s=สมดุล | m=แม่นกว่า
EPOCHS     = 50
BATCH      = 16             # ลดเป็น 8 ถ้า Out of Memory
IMGSZ      = 640
PATIENCE   = 10

print('=' * 60)
print('  🚀 YOLO Fine-tuning เริ่มแล้ว!')
print('=' * 60)
print(f'  Model   : {BASE_MODEL}')
print(f'  Dataset : {yaml_path}')
print(f'  Epochs  : {EPOCHS}')
print(f'  Batch   : {BATCH}')
print('=' * 60)

model = YOLO(BASE_MODEL)

results = model.train(
    data     = yaml_path,
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    patience = PATIENCE,
    device   = 0,
    project  = '/kaggle/working/runs',
    name     = 'traffic_finetune',
    exist_ok = True,
    flipud   = 0.0,
    fliplr   = 0.5,
    mosaic   = 1.0,
)

best_path = '/kaggle/working/runs/traffic_finetune/weights/best.pt'
print(f'\n✅ เทรนเสร็จแล้ว!')
print(f'   Best model: {best_path}')

In [ ]:
# ============================================================
# Step 5: ดูผลลัพธ์ + กราฟ
# ============================================================
import os
from ultralytics import YOLO
from IPython.display import Image, display

best_path   = '/kaggle/working/runs/traffic_finetune/weights/best.pt'
results_dir = '/kaggle/working/runs/traffic_finetune'

if not os.path.exists(best_path):
    print('❌ ไม่พบ best.pt — กรุณารัน Step 4 ก่อน')
else:
    # Validate
    best_model = YOLO(best_path)
    metrics    = best_model.val(data=yaml_path, device=0)

    print('\n' + '=' * 50)
    print('  📊 ผลการเทรน')
    print('=' * 50)
    print(f'  mAP50    : {metrics.box.map50:.4f}   (เป้าหมาย > 0.5)')
    print(f'  mAP50-95 : {metrics.box.map:.4f}')
    print(f'  Precision: {metrics.box.p.mean():.4f}')
    print(f'  Recall   : {metrics.box.r.mean():.4f}')
    print('=' * 50)

    # แสดงกราฟ training
    for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
        img_path = os.path.join(results_dir, img_name)
        if os.path.exists(img_path):
            print(f'\n📈 {img_name}:')
            display(Image(img_path, width=800))

In [ ]:
# ============================================================
# Step 6: เตรียม Output สำหรับ Download
# ============================================================
import os, shutil

best_path = '/kaggle/working/runs/traffic_finetune/weights/best.pt'
copy_path = '/kaggle/working/best.pt'

if os.path.exists(best_path):
    shutil.copy2(best_path, copy_path)
    size_mb = os.path.getsize(copy_path) / 1e6

    print('=' * 60)
    print('  ✅ พร้อม Download!')
    print('=' * 60)
    print(f'  ไฟล์  : {copy_path}')
    print(f'  ขนาด  : {size_mb:.1f} MB')
    print()
    print('  วิธี Download:')
    print('  → กด "Save Version" (มุมขวาบน) → Save & Run All')
    print('  → ไปที่ Output tab → คลิกขวาที่ best.pt → Download')
    print()
    print('  หลัง Download — วาง best.pt ไว้ที่:')
    print('  C:\\Users\\LENOVO\\OneDrive - KASETSART UNIVERSITY\\Documents\\Traffic Project\\')
else:
    print('❌ ไม่พบ best.pt — กรุณารัน Step 4 ก่อน')